# Train YOLOv8n on Penn-Fudan Pedestrian (Colab / Kaggle)

Reproduces the training configuration described in the paper: YOLOv8n, COCO-pretrained,
input resolution 320x320, batch size 8, 38 epochs, fine-tuned on the Penn-Fudan Pedestrian
dataset (person class only).

**On Google Colab:** Runtime -> Change runtime type -> GPU (T4 is fine), then Run All.

**On Kaggle:** create a new Notebook, upload this file (File -> Upload Notebook), then
Settings -> Accelerator -> GPU T4 x2 (or any GPU), then Run All.

Total run time on a free T4: roughly 4-6 minutes for 38 epochs on this small dataset.

## 1. Install dependencies

In [ ]:
!pip install -q ultralytics==8.2.103
import ultralytics
ultralytics.checks()

## 2. Get the dataset

The project already ships a ready-to-train copy at `data/pennfudan_yolo/` -- the real
170-image Penn-Fudan Pedestrian dataset, already converted to YOLO format. Two options:

**Option A (recommended, fastest): upload the bundled dataset directly.**
On your own machine, zip it: `cd data && zip -r pennfudan_yolo.zip pennfudan_yolo`.
Then in Colab/Kaggle's file browser (left sidebar), upload `pennfudan_yolo.zip` to the
notebook's working directory, and run cell 2b below (skip the download cells just below
this one).

**Option B: re-download and rebuild it from scratch inside the notebook instead**
(useful if you don't have the bundled copy, or just want to verify it independently) --
run the next two cells.

In [ ]:
# --- Option A: build the dataset directly inside the notebook ---
import os, random, shutil, zipfile
from pathlib import Path
from urllib.request import urlretrieve
import numpy as np
from PIL import Image

DATASET_URL = "https://www.cis.upenn.edu/~jshi/ped_html/PennFudanPed.zip"
raw_dir = Path("raw"); raw_dir.mkdir(exist_ok=True)
zip_path = raw_dir / "PennFudanPed.zip"
extract_dir = raw_dir / "PennFudanPed"

if not extract_dir.exists():
    print("Downloading Penn-Fudan Pedestrian dataset...")
    urlretrieve(DATASET_URL, zip_path)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(raw_dir)
    print("Done.")
else:
    print("Already downloaded.")

In [ ]:
def mask_to_yolo_boxes(mask_path, img_w, img_h):
    mask = np.array(Image.open(mask_path))
    instance_ids = np.unique(mask)
    instance_ids = instance_ids[instance_ids != 0]
    lines = []
    for inst_id in instance_ids:
        ys, xs = np.where(mask == inst_id)
        if len(xs) == 0:
            continue
        x_min, x_max = xs.min(), xs.max()
        y_min, y_max = ys.min(), ys.max()
        cx = (x_min + x_max) / 2.0 / img_w
        cy = (y_min + y_max) / 2.0 / img_h
        w = (x_max - x_min) / img_w
        h = (y_max - y_min) / img_h
        lines.append(f"0 {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")
    return lines

out_dir = Path("pennfudan_yolo")
png_images_dir = extract_dir / "PNGImages"
ped_masks_dir = extract_dir / "PedMasks"
image_files = sorted(png_images_dir.glob("*.png"))

random.seed(42)
shuffled = image_files[:]
random.shuffle(shuffled)
n = len(shuffled)
n_train = int(n * 0.70)
n_val = int(n * 0.20)
splits = {
    "train": shuffled[:n_train],
    "val": shuffled[n_train:n_train + n_val],
    "test": shuffled[n_train + n_val:],
}

for split_name, files in splits.items():
    (out_dir / "images" / split_name).mkdir(parents=True, exist_ok=True)
    (out_dir / "labels" / split_name).mkdir(parents=True, exist_ok=True)
    for img_path in files:
        mask_path = ped_masks_dir / img_path.name.replace(".png", "_mask.png")
        if not mask_path.exists():
            continue
        img = Image.open(img_path)
        w, h = img.size
        yolo_lines = mask_to_yolo_boxes(mask_path, w, h)
        shutil.copy(img_path, out_dir / "images" / split_name / img_path.name)
        (out_dir / "labels" / split_name / img_path.name.replace(".png", ".txt")).write_text("\n".join(yolo_lines))
    print(f"[{split_name}] {len(files)} images")

data_yaml_path = out_dir / "data.yaml"
data_yaml_path.write_text(f"""path: {out_dir.resolve()}
train: images/train
val: images/val
test: images/test

names:
  0: person
""")
print("data.yaml written to", data_yaml_path)

## 2b. Using the bundled dataset (Option A) -- run this cell instead of the two above

In [ ]:
# Run this cell if you uploaded pennfudan_yolo.zip (Option A). It also fixes the
# data.yaml 'path:' line to match this notebook's working directory, the same way
# data/fix_data_yaml.py does locally -- necessary because the zip was built on a
# different machine.
import zipfile
from pathlib import Path

zip_candidate = Path("pennfudan_yolo.zip")
if zip_candidate.exists():
    with zipfile.ZipFile(zip_candidate, "r") as zf:
        zf.extractall(".")
    data_yaml_path = Path("pennfudan_yolo/data.yaml")
    lines = data_yaml_path.read_text().splitlines()
    fixed = [
        f"path: {Path('pennfudan_yolo').resolve().as_posix()}" if l.startswith("path:") else l
        for l in lines
    ]
    data_yaml_path.write_text("\n".join(fixed) + "\n")
    print(f"Extracted and fixed {data_yaml_path}")
else:
    print("pennfudan_yolo.zip not found -- either upload it (Option A), or run the two "
          "cells above instead (Option B) to rebuild the dataset from scratch.")

## 3. Train

Same hyperparameters as the paper: 320x320 input, batch size 8, 38 epochs, YOLOv8n, COCO-pretrained.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")  # COCO-pretrained checkpoint

results = model.train(
    data=str(data_yaml_path),
    imgsz=320,
    batch=8,
    epochs=38,
    patience=0,          # paper trains the full 38 epochs, no early stopping
    project="runs/detect",
    name="train",
    exist_ok=True,
    classes=[0],          # restrict to person class only, matches the paper
    augment=True,
    fliplr=0.5,            # horizontal flip augmentation
    degrees=10.0,          # bounded rotation augmentation
    hsv_h=0.015, hsv_s=0.5, hsv_v=0.3,  # photometric colour jitter
)

## 4. Evaluate on the held-out test split

In [ ]:
metrics = model.val(data=str(data_yaml_path), split="test", imgsz=320)
print("Precision:", metrics.box.mp)
print("Recall:", metrics.box.mr)
print("mAP@0.5:", metrics.box.map50)
print("mAP@0.5:0.95:", metrics.box.map)

## 5. Download the trained weights

Zips `best.pt` for download. On Colab this pops a browser download; on Kaggle, find it
under the Output tab of the notebook after committing/running, or use the file browser.

In [ ]:
import shutil
best_path = "runs/detect/train/weights/best.pt"
shutil.copy(best_path, "best.pt")
print("Weights ready at ./best.pt -- download this file and place it in your project at weights/best.pt")

try:
    from google.colab import files
    files.download("best.pt")
except ImportError:
    print("Not running on Colab -- on Kaggle, grab best.pt from the notebook's Output/Data pane instead.")